# PTBR-Reranker — Phase 1 validation (Kaggle, free tier)

Valida o pipeline de Phase 1 sem custo de GPU paga:

1. Clona o repositório `tardellirs/ptbr-reranker` (público).
2. Instala dependências (cache do Kaggle acelera reruns).
3. Roda `python data/download_mmarco.py --small` — baixa ~10k passagens + 1k train queries + 100 dev queries do mMARCO-PT, gravando `manifest.json` com SHA da revisão `refs/convert/parquet`.
4. Roda `pytest -m slow tests/test_data_pipeline.py::test_albertina_loads_and_predicts` — confirma que `PORTULAN/albertina-100m-portuguese-ptbr-encoder` carrega via `sentence_transformers.CrossEncoder` e produz scores em CPU.
5. Inspeciona o manifest para auditoria do paper.

**Settings deste Kernel (Kaggle UI > Settings):**
- Accelerator: **None** (CPU). Tudo aqui roda em CPU; não gaste cota de GPU.
- Internet: **On** (precisa para baixar do HuggingFace; conta precisa ter telefone verificado).
- Persistence: **Variables and files** (opcional, para reusar entre runs).
- Language: Python 3.

Execution time esperado: 5–10 min (depende de cache do HF e velocidade de download).

> Nota: MIRACL não tem subset PT (idiomas disponíveis: ar/bn/es/fa/fi/hi/id/ja/ko/sw/te/th/yo/zh). Avaliação cross-domain a definir em sessão futura.

## 1. Clonar repositório e instalar dependências

In [ ]:
!rm -rf /kaggle/working/ptbr-reranker
!git clone --depth 1 https://github.com/tardellirs/ptbr-reranker.git /kaggle/working/ptbr-reranker
%cd /kaggle/working/ptbr-reranker
!git log --oneline -1

In [ ]:
# Install dev extras. Kaggle base image already has torch/transformers/datasets,
# so this should be fast (mostly pytest, ruff, mypy, sentence-transformers, codecarbon).
!pip install -q -e ".[dev]" 2>&1 | tail -5

## 2. Download em modo `--small`

Materializa ~10k passagens, 1k queries train, 100 queries dev, e MIRACL-PT dev. Grava `data/raw/manifest.json` com a SHA da revisão HF — o artefato direto que vai para `docs/reproducibility.md` no momento do release.

In [ ]:
!python data/download_mmarco.py --small

In [ ]:
import json
from pathlib import Path

manifest = json.loads(Path("data/raw/manifest.json").read_text())
print(json.dumps(manifest, indent=2, ensure_ascii=False))

In [ ]:
# Validar contagens (deve retornar exit code 0).
!python data/download_mmarco.py --check --small

## 3. Sanity check dos dados baixados

Uma olhada rápida no formato das passagens e queries — confirma que está em PT-BR mesmo (não em inglês original).

In [ ]:
import pandas as pd

for split_path in sorted(Path("data/raw/mmarco").glob("*.parquet")):
    df = pd.read_parquet(split_path)
    print(f"=== {split_path.name} ({len(df):,} rows) ===")
    print(df.head(3).to_string(max_colwidth=120))
    print()

## 4. Smoke test do Albertina-100m

Carrega o foundation model via `sentence_transformers.CrossEncoder` e scoreia 2 pares. O modelo ainda não foi fine-tuned como reranker, então os scores não precisam fazer sentido — só queremos confirmar que o caminho de loading e inferência funciona antes de pagar GPU.

In [ ]:
!pytest -v -m slow tests/test_data_pipeline.py::test_albertina_loads_and_predicts 2>&1 | tail -20

In [ ]:
# Também roda toda a suite rápida para confirmar que nada quebrou.
!pytest -v -m "not gpu and not slow" --tb=short 2>&1 | tail -30

## 5. Inspeção qualitativa de um par (query, passage)

In [ ]:
import torch
from sentence_transformers import CrossEncoder

# Force float32 — Kaggle CPU image otherwise mixes bfloat16 weights with
# float32 inputs in DeBERTa attention, raising a dtype mismatch RuntimeError.
model = CrossEncoder(
    "PORTULAN/albertina-100m-portuguese-ptbr-encoder",
    num_labels=1,
    max_length=128,
    model_kwargs={"torch_dtype": torch.float32},
)
model.model.float()

query = "qual a capital do Brasil?"
passages = [
    "Brasília é a capital federal do Brasil desde 1960.",
    "São Paulo é a maior cidade do Brasil.",
    "Receita de bolo de chocolate com cobertura de brigadeiro.",
]
scores = model.predict([(query, p) for p in passages])
for passage, score in sorted(zip(passages, scores.tolist(), strict=True), key=lambda x: -x[1]):
    print(f"{score:+.4f}  {passage}")

print("\nNota: o modelo NÃO foi fine-tuned ainda. Scores não refletem relevância real;")
print("este é apenas o smoke test do pipeline. Phase 3 (treino) muda isso.")

## 6. Checklist final

Se todas as células acima passaram:

- [x] mMARCO-PT `--small` baixado e validado.
- [x] MIRACL-PT dev baixado.
- [x] Manifest com SHA HF registrado.
- [x] Albertina-100m carrega via CrossEncoder e produz scores em CPU.
- [x] Suite de testes não-slow passa.

O pipeline de Phase 1 está validado. Próximos passos:

1. Decidir se hospedo os hard negatives como dataset HF separado (`tardellirs/mmarco-ptbr-hardnegatives`).
2. Implementar `data/mine_hard_negatives.py` (precisará GPU — Kaggle T4×2 free é suficiente para o `--small`).
3. Migrar para Runpod Community A100 SXM para o treino full em mMARCO-PT inteiro.

Registrar resultado deste run no `docs/lab_notebook.md` do repositório (entrada datada com observações).